# Zero-shot Segmentation với DINOv2 — Demo

Notebook minh hoạ pipeline **training-free segmentation** trên Oxford-IIIT Pet.

Hai phương pháp:
1. **Prototype-based** — dùng 1–N ảnh tham chiếu cho mỗi class (kèm mask) → tính prototype patch-feature → cosine match.
2. **Unsupervised clustering** — KMeans trên patch features, gán nhãn cluster thủ công.

Trước khi chạy: `bash scripts/download_oxford_pet.sh` để tải dataset.

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

# Cho phép import zero_shot_seg từ ../src bất kể notebook được mở từ đâu.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'src' / 'zero_shot_seg').exists():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('REPO_ROOT =', REPO_ROOT)
print('SRC       =', SRC)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from zero_shot_seg import (
    DINOv2DenseExtractor,
    PrototypeZeroShotSegmenter,
    ClusteringSegmenter,
    OXFORD_PET_TRIMAP_CLASSES,
)
from zero_shot_seg.data import OxfordPetSegDataset, denormalize
from zero_shot_seg.visualize import (
    DEFAULT_PALETTE, labels_to_color, overlay_segmentation, legend_entries,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)
print('classes =', OXFORD_PET_TRIMAP_CLASSES)

## 2. Tải DINOv2 (backbone đông cứng)

Lần đầu chạy: torch.hub sẽ download weights. CPU dùng `dinov2_vits14` (~21M params) để chạy được trong vài giây/ảnh.

In [ ]:
extractor = DINOv2DenseExtractor(model_name='dinov2_vits14', device=DEVICE)
print('patch_size =', extractor.patch_size, '| embed_dim =', extractor.embed_dim)

## 3. Dataset

In [ ]:
DATA_ROOT = REPO_ROOT / 'data' / 'oxford_pet'
assert DATA_ROOT.exists(), f'Chưa tải dataset. Hãy chạy: bash scripts/download_oxford_pet.sh'

IMAGE_SIZE = 224  # bội số của 14
dataset = OxfordPetSegDataset(str(DATA_ROOT), image_size=IMAGE_SIZE, max_samples=12)
print(f'Số mẫu: {len(dataset)}')
img, mask, name = dataset[0]
print('img:', img.shape, 'mask:', mask.shape, 'name:', name)

In [ ]:
# Quan sát vài ảnh + trimap ground truth
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    img, mask, name = dataset[i]
    axes[0, i].imshow(denormalize(img)); axes[0, i].set_title(name); axes[0, i].axis('off')
    axes[1, i].imshow(labels_to_color(mask)); axes[1, i].set_title('trimap GT'); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

## 4. Prototype-based zero-shot segmentation

Lấy `NUM_REF` ảnh đầu tiên làm reference. Với mỗi class, lọc các patch thuộc class đó (theo trimap), trung bình → prototype.

In [ ]:
NUM_REF = 3
references = {c: [] for c in OXFORD_PET_TRIMAP_CLASSES}
for i in range(NUM_REF):
    img, mask, _ = dataset[i]
    for cls_idx, cls in enumerate(OXFORD_PET_TRIMAP_CLASSES):
        cls_mask = (mask == cls_idx)
        if cls_mask.sum() > 0:
            references[cls].append((img, cls_mask))
for k, v in references.items():
    print(f'  {k}: {len(v)} reference(s)')

segmenter = PrototypeZeroShotSegmenter(extractor, OXFORD_PET_TRIMAP_CLASSES)
segmenter.build_prototypes(references)
print('prototype shape:', tuple(segmenter.prototypes.shape))

In [ ]:
def mean_iou(pred, target, num_classes):
    p = pred.flatten(); t = target.flatten(); ious = []
    for c in range(num_classes):
        pm, tm = (p == c), (t == c)
        u = (pm | tm).sum().item()
        if u == 0: continue
        ious.append((pm & tm).sum().item() / u)
    return float(np.mean(ious)) if ious else 0.0

TEST_START, TEST_N = NUM_REF, 4
fig, axes = plt.subplots(TEST_N, 3, figsize=(11, 3 * TEST_N))
mious = []
for row, i in enumerate(range(TEST_START, TEST_START + TEST_N)):
    img, mask, name = dataset[i]
    pred = segmenter.segment(img).cpu()
    miou = mean_iou(pred, mask, len(OXFORD_PET_TRIMAP_CLASSES))
    mious.append(miou)
    rgb = denormalize(img)
    axes[row, 0].imshow(rgb); axes[row, 0].set_title(f'{name}'); axes[row, 0].axis('off')
    axes[row, 1].imshow(overlay_segmentation(rgb, mask)); axes[row, 1].set_title('GT'); axes[row, 1].axis('off')
    axes[row, 2].imshow(overlay_segmentation(rgb, pred)); axes[row, 2].set_title(f'pred (mIoU={miou:.2f})'); axes[row, 2].axis('off')

handles = [Patch(facecolor=np.array(c)/255, label=name) for name, c in legend_entries(OXFORD_PET_TRIMAP_CLASSES)]
fig.legend(handles=handles, loc='lower center', ncol=len(OXFORD_PET_TRIMAP_CLASSES), bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.show()
print(f'mean mIoU = {np.mean(mious):.3f}')

## 5. Unsupervised clustering segmentation

Không cần reference: gom toàn bộ patch features từ một số ảnh, KMeans (cosine) → các cluster ID.

In [ ]:
K = 4  # số cluster — có thể > số class thực để cho phép phân nhỏ.
FIT_IMAGES = [dataset[i][0] for i in range(6)]
cluster_seg = ClusteringSegmenter(extractor, num_clusters=K)
cluster_seg.fit(FIT_IMAGES, n_iter=20)
print('centroids shape:', tuple(cluster_seg.centroids.shape))

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(9, 11))
for row, i in enumerate(range(6, 9)):
    img, _, name = dataset[i]
    pred = cluster_seg.segment(img).cpu()
    rgb = denormalize(img)
    axes[row, 0].imshow(rgb); axes[row, 0].set_title(name); axes[row, 0].axis('off')
    axes[row, 1].imshow(overlay_segmentation(rgb, pred)); axes[row, 1].set_title(f'cluster map ({K} clusters)'); axes[row, 1].axis('off')
plt.tight_layout(); plt.show()

## 6. Ghi chú
- DINOv2 không có alignment text–image như CLIP. "Zero-shot" ở đây hiểu theo nghĩa **training-free** — không update trọng số nào.
- Prototype-based chính xác hơn nhưng cần ít nhất 1 ảnh reference per class.
- Clustering thuần unsupervised — cần gán nhãn cluster thủ công.
- Để cải thiện: tăng `image_size` (lên 392 = 14·28), dùng backbone lớn hơn (ViT-B/L), hoặc kết hợp với CRF/SLIC post-processing.